# GPU-Accelerated UMAP with cuML
This notebook is designed to be run in Google Colab with a GPU. 

**Important**: Before running, go to `Runtime > Change runtime type` and select `T4 GPU` (or better).

In [ ]:
# Install RAPIDS cuML (GPU accelerated machine learning library)
!pip install cudf-cu12 cuml-cu12 --extra-index-url=https://pypi.nvidia.com

In [ ]:
import pandas as pd
import numpy as np
import cuml
from sklearn.preprocessing import StandardScaler
from google.colab import files

# Upload your processed Parquet file
print("Please upload your 'features_pre_umap.parquet' file:")
uploaded = files.upload()
filename = list(uploaded.keys())[0]
print(f"Loaded {filename}")

In [ ]:
df = pd.read_parquet(filename)

# The features we use for UMAP in your local features.py
umap_features = [
    'Distance', 'DepDelay_Log', 'ArrDelay_Log', 
    'Airline_Delay_Risk', 'Origin_Airport_Risk', 'Dest_Airport_Risk', 
    'Ground_Time_Ratio', 'Time_Made_Up', 
    'Origin_Dep_Congestion', 'Dest_Arr_Congestion'
]

# If DepDelay_Log and ArrDelay_Log don't exist in the data yet, calculate them:
if 'ArrDelay_Log' not in df.columns:
    df['ArrDelay_Log'] = np.log1p(df['ArrDelay'].clip(lower=0))
if 'DepDelay_Log' not in df.columns:
    df['DepDelay_Log'] = np.log1p(df['DepDelay'].clip(lower=0))

print("Preparing features...")
X_umap = df[umap_features].fillna(0).values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_umap)

print("Running cuML UMAP on GPU (this should take a few minutes or less)...")
reducer = cuml.UMAP(n_components=3, n_neighbors=15, min_dist=0.15, random_state=42)
components = reducer.fit_transform(X_scaled)

print("Adding UMAP components to the dataframe...")
df['UMAP_1'] = components[:, 0]
df['UMAP_2'] = components[:, 1]
df['UMAP_3'] = components[:, 2]

output_filename = "processed_flights_with_umap.parquet"
print(f"Saving results to {output_filename}...")
df.to_parquet(output_filename, index=False, compression='snappy')

print("Downloading results back to your local machine...")
files.download(output_filename)